<a href="https://colab.research.google.com/github/maheshisewwandisamaraweera/image-authentication-project/blob/maheshi/correct_image_edit_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q diffusers transformers accelerate safetensors pillow matplotlib

import os
import torch
from PIL import Image
import matplotlib.pyplot as plt
from diffusers import StableDiffusionInpaintPipeline

# ---------------- PATHS ----------------
input_folder = "/content/drive/MyDrive/image-authentication-project/data/actual"
output_folder = "/content/drive/MyDrive/image-authentication-project/data/edited"

os.makedirs(output_folder, exist_ok=True)

# ---------------- LOAD MODEL ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
).to(device)

pipe.enable_attention_slicing()

# ---------------- PROMPTS ----------------
prompts = {
    "fight": (
        "realistic edited photo, same scene, same background, same people, "
        "people standing apart calmly, no fighting, no violence, natural CCTV quality"
    ),
    "accident": (
        "realistic edited photo, same road, same camera angle, no accident, "
        "normal traffic scene, clear road, no crash, no damaged vehicles, CCTV quality"
    ),
    "weapon": (
        "realistic edited photo, same scene, same person, empty hands, "
        "no weapon, calm situation, natural CCTV quality"
    ),
    "fire": (
        "realistic edited photo, same scene, no fire, no smoke, "
        "normal safe environment, natural CCTV quality"
    )
}

negative_prompt = (
    "crime, fight, violence, accident, crash, weapon, fire, smoke, blood, injury, "
    "ai generated, fake, cartoon, painting, distorted, blurry, extra limbs, bad quality"
)

# ---------------- PROCESS IMAGES ONE BY ONE ----------------
image_exts = (".jpg", ".jpeg", ".png", ".webp")

for file_name in os.listdir(input_folder):

    if not file_name.lower().endswith(image_exts):
        continue

    image_path = os.path.join(input_folder, file_name)
    image = Image.open(image_path).convert("RGB")
    original_size = image.size

    # Show image
    plt.figure(figsize=(7,5))
    plt.imshow(image)
    plt.title(file_name)
    plt.axis("off")
    plt.show()

    print("Choose type for this image:")
    print("1 = fight")
    print("2 = accident")
    print("3 = weapon")
    print("4 = fire")
    print("s = skip")

    choice = input("Enter choice: ").strip()

    if choice == "1":
        scenario = "fight"
    elif choice == "2":
        scenario = "accident"
    elif choice == "3":
        scenario = "weapon"
    elif choice == "4":
        scenario = "fire"
    elif choice.lower() == "s":
        print("Skipped:", file_name)
        continue
    else:
        print("Invalid choice, skipped:", file_name)
        continue

    # Full-image mask
    mask = Image.new("L", image.size, 255)

    SD_SIZE = 512
    img_sd = image.resize((SD_SIZE, SD_SIZE))
    mask_sd = mask.resize((SD_SIZE, SD_SIZE))

    result = pipe(
        prompt=prompts[scenario],
        negative_prompt=negative_prompt,
        image=img_sd,
        mask_image=mask_sd,
        num_inference_steps=30,
        guidance_scale=5.5,
        strength=0.40
    ).images[0]

    result = result.resize(original_size)

    # Show result
    plt.figure(figsize=(7,5))
    plt.imshow(result)
    plt.title("Edited Non-Crime Result")
    plt.axis("off")
    plt.show()

    output_path = os.path.join(output_folder, file_name)
    result.save(output_path)

    print("Saved:", output_path)
    print("-" * 50)

print("All selected images completed.")